In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace('[\\[\\]]', '', regex=True)
#selecting only the required columns
columns_to_keep = [
    'quote_date','underlying_last', 'expire_date', 'strike',
    'c_bid', 'c_ask', 'c_last', 'c_iv','c_delta','c_vega',
    'p_bid', 'p_ask', 'p_last', 'p_iv','c_gamma','c_theta',
    'p_theta','p_gamma',
    'p_vega', 'p_delta','p_rho',
    'c_rho'
]
aapl = df[columns_to_keep].copy()
aapl['expire_date']=pd.to_datetime(aapl['expire_date'])
aapl['quote_date']=pd.to_datetime(aapl['quote_date'])
aapl.sort_values(by=['quote_date', 'expire_date', 'strike'], inplace=True)


In [ ]:

open_positions = []
max_open_positions = 10
initial_capital = 100000
capital = initial_capital
capital_history = []         
trade_pnls = []              
open_position_counts = []    
dates_tracked = []          

numeric_cols = ['strike', 'c_bid', 'c_ask', 'p_bid', 'p_ask', 'c_last', 'p_last', 'c_iv', 'p_iv']
aapl[numeric_cols] = aapl[numeric_cols].apply(pd.to_numeric, errors='coerce')
quote_dates = sorted(aapl['quote_date'].unique())

for date in quote_dates:
    # Close expired positions first
    expired_positions = [pos for pos in open_positions if pos['expiry'] == date]
    for pos in expired_positions:
        expiry_data = aapl[
            (aapl['quote_date'] == date) &
            (aapl['expire_date'] == date)
        ]
        if expiry_data.empty:
            continue 
        expiry_price = expiry_data['underlying_last'].iloc[0]

        if expiry_price > pos['buy_strike']:
            intrinsic_value = expiry_price - pos['buy_strike']
            pnl = intrinsic_value * 100  # Gain from exercising the option as it is In The Money (ITM)
        else:
            pnl = 0  # Option expired worthless
        capital += pnl
        trade_pnls.append(pnl)
        print(f"[{date}] Position {pos['id']} Exit:")
        print(f"  Expired Spot: {expiry_price}")
        print(f"  Buy {pos['buy_strike']}, Initial Cost: {pos['cost']:.2f}")
        print(f"  PnL from exercise: {pnl:.2f}, Capital: {capital:.2f}")
        print("-" * 80)
    # Remove closed positions
    open_positions = [pos for pos in open_positions if pos['expiry'] != date]
    # Record portfolio capital and open positions
    capital_history.append(capital)
    open_position_counts.append(len(open_positions))
    dates_tracked.append(date)

    if len(open_positions) >= max_open_positions:
        continue  

    spot_price = aapl[aapl['quote_date'] == date]['underlying_last'].iloc[0]
    entered_trade = False

    for expiry in sorted(aapl[aapl['expire_date'] > date]['expire_date'].unique()):
        df = aapl[(aapl['quote_date'] == date) & (aapl['expire_date'] == expiry)]
        calls = df[['strike', 'c_bid', 'c_ask']].dropna().sort_values('strike').reset_index(drop=True)

        sell_candidates = calls[calls['strike'] < spot_price]
        if sell_candidates.empty:
            continue
        sell_strike = sell_candidates['strike'].iloc[-1]
        buy_strike = sell_strike - 5

        buy_row = calls[calls['strike'] == buy_strike]
        sell_row = calls[calls['strike'] == sell_strike]

        if buy_row.empty or sell_row.empty:
            continue

        buy_ask = buy_row['c_ask'].iloc[0]
        sell_bid = sell_row['c_bid'].iloc[0]

        if pd.isna(buy_ask) or pd.isna(sell_bid):
            continue

        cost = buy_ask - sell_bid
        if cost <= 0:
            continue

        max_profit = 10 - cost
        rr = max_profit / cost

        position_number = len(open_positions) + 1
        open_positions.append({
            'id': position_number,
            'date': date,
            'expiry': expiry,
            'buy_strike': buy_strike,
            'buy_ask': buy_ask,
            'sell_strike': sell_strike,
            'sell_bid': sell_bid,
            'cost': cost,
            'max_profit': max_profit,
            'rr': rr
        })

        capital -= cost * 100
        print(f"[{date}] Position {position_number} Entry:")
        print(f"  Buy {buy_strike} @ {buy_ask}, Sell {sell_strike} @ {sell_bid}")
        print(f"  Expiry: {expiry}, Cost: {cost:.2f}, Max Profit: {max_profit:.2f}, RR: {rr:.2f}")
        print(f"  Capital after trade: {capital:.2f}")
        print("-" * 80)
        break  # Only one trade per day

plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
plt.plot(dates_tracked, capital_history, label='Capital', color='blue')
plt.title("Capital Over Time")
plt.xlabel("Date")
plt.ylabel("Capital")
plt.xticks(rotation=45)
plt.grid(True)

plt.subplot(1, 3, 2)
plt.hist(trade_pnls, bins=20, color='green', edgecolor='black')
plt.title("Distribution of Trade PnLs")
plt.xlabel("PnL")
plt.ylabel("Frequency")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(dates_tracked, open_position_counts, color='orange', label='Open Positions')
plt.title("Open Positions Over Time")
plt.xlabel("Date")
plt.ylabel("# of Positions")
plt.xticks(rotation=45)
plt.grid(True)

plt.tight_layout()
plt.show()
